In [72]:
from tvm.script import tir as T


@T.prim_func
def main(
    A: T.Buffer(128, "float32"),
    B: T.Buffer(128, "float32"),
    C: T.Buffer(128, "float32"),
):
    for i in range(128):
        with T.sblock("C"):
            vi = T.axis.spatial(128, i)
            C[vi] = A[vi] + B[vi]

In [73]:
print(main.script())

# from tvm.script import tir as T

@T.prim_func
def main(A: T.Buffer((128,), "float32"), B: T.Buffer((128,), "float32"), C: T.Buffer((128,), "float32")):
    # with T.sblock("root"):
    for i in range(128):
        with T.sblock("C"):
            vi = T.axis.spatial(128, i)
            T.reads(A[vi], B[vi])
            T.writes(C[vi])
            C[vi] = A[vi] + B[vi]


In [74]:
import tvm

sch = tvm.s_tir.Schedule(main)
print(sch.mod.script())

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(A: T.Buffer((128,), "float32"), B: T.Buffer((128,), "float32"), C: T.Buffer((128,), "float32")):
        # with T.sblock("root"):
        for i in range(128):
            with T.sblock("C"):
                vi = T.axis.spatial(128, i)
                T.reads(A[vi], B[vi])
                T.writes(C[vi])
                C[vi] = A[vi] + B[vi]


In [75]:
ex = tvm.compile(main, target="llvm")
# print(ex.mod.inspect_source())

In [76]:
import time

import numpy as np
from numpy.typing import NDArray


def lnumpy_mm_relu_ijk(
    A: NDArray[np.float64], B: NDArray[np.float64], C: NDArray[np.float64]
):
    Y = np.empty((128, 128), dtype="float64")
    for i in range(128):
        for j in range(128):
            for k in range(128):
                if k == 0:
                    Y[i, j] = 0
                Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
        for i in range(128):
            for j in range(128):
                C[i, j] = max(Y[i, j], 0)


def lnumpy_mm_relu_ikj(
    A: NDArray[np.float64], B: NDArray[np.float64], C: NDArray[np.float64]
):
    Y = np.empty((128, 128), dtype="float64")
    for i in range(128):
        for k in range(128):
            if A[i, k] == 0:
                continue
            for j in range(128):
                Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
        for i in range(128):
            for j in range(128):
                C[i, j] = max(Y[i, j], 0)


dtype = "float32"
a_np = np.random.rand(128, 128).astype(dtype)
b_np = np.random.rand(128, 128).astype(dtype)
c_np = np.empty((128, 128), dtype=dtype)

t0 = time.perf_counter()
lnumpy_mm_relu_ijk(a_np, b_np, c_np)
t1 = time.perf_counter()
lnumpy_mm_relu_ikj(a_np, b_np, c_np)
t2 = time.perf_counter()

print(f"ijk: {t1 - t0:.3f}s")
print(f"ikj: {t2 - t1:.3f}s")


ijk: 2.056s
ikj: 2.028s


In [77]:
dtype = "float32"


@tvm.script.ir_module
class MyModule:
    @T.prim_func
    def mm_relu(
        A: T.Buffer((128, 128), dtype),
        B: T.Buffer((128, 128), dtype),
        C: T.Buffer((128, 128), dtype),
    ):
        T.func_attr({"global_symbol": "mm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((128, 128), dtype)
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                vk = T.axis.reduce(128, k)
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


@tvm.script.ir_module
class MyModuleIKJ:
    @T.prim_func
    def mm_relu(
        A: T.Buffer((128, 128), dtype),
        B: T.Buffer((128, 128), dtype),
        C: T.Buffer((128, 128), dtype),
    ):
        T.func_attr({"global_symbol": "mm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((128, 128), dtype)
        for i, k, j in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                vk = T.axis.reduce(128, k)
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


In [78]:
a_np = np.random.rand(n, n).astype("float32")
b_np = np.random.rand(n, n).astype("float32")
c_np = np.zeros((n, n), dtype="float32")

In [79]:
ex = tvm.compile(MyModule, target="llvm")
ev = ex.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev(a_np, b_np, c_np))


Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   2.1250       2.1250       2.1250       2.1250       0.0000                  


In [80]:
ex_ikj = tvm.compile(MyModuleIKJ, target="llvm")
ev_ijk = ex_ikj.mod.time_evaluator("mm_relu", tvm.cpu(), number=100)
print(ev_ijk(a_np, b_np, c_np))


Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   0.2023       0.2023       0.2023       0.2023       0.0000                  
